In [ ]:
# 1. INSTALAÇÃO DAS DEPENDÊNCIAS
!pip install flask flask-cors firebase-admin tensorflow pandas numpy joblib scikit-learn pyngrok

import os
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
from flask import Flask, jsonify, request
from flask_cors import CORS
import firebase_admin
from firebase_admin import credentials, db
from google.colab import files
from pyngrok import ngrok

# 2. CARREGAR ARQUIVOS NECESSÁRIOS (serviceAccountKey.json, modelo_lstm_energia.keras, scaler_energia.pkl)
print("Por favor, faz o upload dos ficheiros necessários (.json, .keras, .pkl):")
uploaded = files.upload()

# 3. CONFIGURAÇÃO E AUTENTICAÇÃO DO NGROK
# Subtitui pelo teu token caso este expire
ngrok.set_auth_token("3E6FIuJGibkJKDB6xQoe9mEXqRW_6sfmiqdyYkc8xfB36qBnk")

# 4. INICIALIZAÇÃO DO FLASK E CONFIGURAÇÃO DE CORS
app = Flask(__name__)
CORS(app)  # Permite que o teu Front-end faça requisições sem bloqueios de segurança

# 5. CONEXÃO COM O FIREBASE REALTIME DATABASE
if not firebase_admin._apps:
    cred = credentials.Certificate("serviceAccountKey.json")
    firebase_admin.initialize_app(cred, {
        'databaseURL': 'https://consumo-conciente-default-rtdb.firebaseio.com/'
    })

# 6. CARREGAMENTO DOS MODELOS DE INTELIGÊNCIA ARTIFICIAL
MODELO_PATH = 'modelo_lstm_energia.keras'
SCALER_PATH = 'scaler_energia.pkl'

print("A carregar o modelo LSTM e o Scaler...")
modelo = tf.keras.models.load_model(MODELO_PATH)
scaler = joblib.load(SCALER_PATH)
print("Modelos carregados com sucesso!")

# --- CONFIGURAÇÕES DO DATASET REAL ---
# Definindo as variáveis com os nomes exatos do teu banco Firebase
COLUNAS_MONITORADAS = ['consumo_total_pzem', 'corrente', 'potencia', 'voltagem']
INDICE_ALVO = COLUNAS_MONITORADAS.index('consumo_total_pzem')
JANELA_PADRAO = 60

# 7. FUNÇÕES AUXILIARES (TRATAMENTO DE DADOS E PREVISÃO)
def buscar_dados_pzem(quantidade):
    """Busca o histórico do Firebase e aplana a estrutura hierárquica (Data -> Hora)"""
    ref = db.reference('leituras')
    dados = ref.get()

    if not dados:
        return pd.DataFrame()

    linhas_achatadas = []

    # Ordena as datas para garantir a ordem cronológica correta
    for data in sorted(dados.keys()):
        horas = dados[data]
        if isinstance(horas, dict):
            # Ordena as horas dentro de cada data
            for hora in sorted(horas.keys()):
                atributos = horas[hora]
                # Filtro importante: Garante que estamos a ler o dicionário da leitura e ignora metadados soltos
                if isinstance(atributos, dict):
                    registro = atributos.copy()

                    # Sistema de tolerância (fallbacks) caso existam chaves antigas no banco
                    if 'consumo' in registro and 'consumo_total_pzem' not in registro:
                        registro['consumo_total_pzem'] = registro['consumo']
                    elif 'consumo_acumulado' in registro and 'consumo_total_pzem' not in registro:
                        registro['consumo_total_pzem'] = registro['consumo_acumulado']

                    linhas_achatadas.append(registro)

    # Cria o DataFrame a partir da lista plana
    df = pd.DataFrame(linhas_achatadas)

    if df.empty:
        return df

    # Validação: Garante que todas as colunas exigidas existem no DataFrame
    for col in COLUNAS_MONITORADAS:
        if col not in df.columns:
            df[col] = 0.0

    # Filtra apenas as colunas que a rede LSTM conhece e retorna as últimas N amostras
    df_filtrado = df[COLUNAS_MONITORADAS].dropna()
    return df_filtrado.tail(quantidade)


def prever_passos_futuros(df, passos_solicitados):
    """Aplica o modelo LSTM de forma iterativa para prever N passos à frente"""
    dados_normalizados = scaler.transform(df[COLUNAS_MONITORADAS])
    janela_atual = dados_normalizados.copy()
    previsoes_finais = []

    for _ in range(passos_solicitados):
        # Ajusta o formato para o padrão exigido pelo Keras/TensorFlow: (1, Janela, 4)
        input_lstm = np.reshape(janela_atual, (1, janela_atual.shape[0], janela_atual.shape[1]))

        # Faz a predição do próximo ponto (retorna o valor escalonado)
        predicao_escalonada = modelo.predict(input_lstm, verbose=0)

        # Cria uma linha temporária para aplicar a inversão do scaler de forma correta
        linha_fake = np.zeros((1, len(COLUNAS_MONITORADAS)))
        linha_fake[0, INDICE_ALVO] = predicao_escalonada[0, 0]

        # Desnormaliza o valor obtido para voltar à escala real (kWh)
        valor_real_consumo = scaler.inverse_transform(linha_fake)[0, INDICE_ALVO]
        previsoes_finais.append(float(valor_real_consumo))

        # --- Atualização da Janela Temporal ---
        proxima_linha = janela_atual[-1].copy()
        proxima_linha[INDICE_ALVO] = predicao_escalonada[0, 0]

        # Remove o primeiro registo (mais antigo) e insere o novo no fim da matriz
        janela_atual = np.vstack([janela_atual[1:], proxima_linha])

    return previsoes_finais

# --- 8. ENDPOINTS DA API REST ---

@app.route('/api/status_atual', methods=['GET'])
def obter_status_atual():
    """Retorna os dados em tempo real mais recentes do sensor e a previsão imediata (+1 passo)"""
    df = buscar_dados_pzem(JANELA_PADRAO)

    if len(df) < JANELA_PADRAO:
        return jsonify({
            'erro': f'Histórico insuficiente no Firebase. São necessárias {JANELA_PADRAO} amostras, mas o sistema possui apenas {len(df)}.'
        }), 400

    ultima_leitura = df.iloc[-1].to_dict()
    previsao_imediata = prever_passos_futuros(df, passos_solicitados=1)

    return jsonify({
        'status': 'sucesso',
        'leitura_atual': ultima_leitura,
        'previsao_proximo_passo': previsao_imediata[0]
    })


@app.route('/api/previsao_futura', methods=['GET'])
def obter_previsao_futura():
    """Retorna a projeção de N passos à frente definida dinamicamente pelo parâmetro '?passos=' na URL"""
    passos = int(request.args.get('passos', 5))
    df = buscar_dados_pzem(JANELA_PADRAO)

    if len(df) < JANELA_PADRAO:
        return jsonify({'erro': 'Histórico insuficiente para realizar previsões futuras.'}), 400

    projecao = prever_passos_futuros(df, passos_solicitados=passos)

    return jsonify({
        'status': 'sucesso',
        'passos_calculados': passos,
        'valores_projetados': projecao
    })


@app.route('/', methods=['GET'])
def testar_servidor():
    return jsonify({'mensagem': 'API de Eficiência Energética a rodar com sucesso!'})

# --- 9. EXECUÇÃO DO TUNEL E SERVIDOR ---
if __name__ == '__main__':
    # Inicia a conexão pública com o Ngrok
    public_url = ngrok.connect(5000)
    print("=" * 60)
    print("URL PÚBLICA DA API (Utiliza esta no teu Front-end):")
    print(public_url)
    print("=" * 60)

    # Executa o servidor Flask localmente na porta 5000
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )

Por favor, faz o upload dos ficheiros necessários (.json, .keras, .pkl):


Saving modelo_lstm_energia.keras to modelo_lstm_energia.keras
Saving scaler_energia.pkl to scaler_energia.pkl
Saving serviceAccountKey.json to serviceAccountKey.json
A carregar o modelo LSTM e o Scaler...
Modelos carregados com sucesso!
URL PÚBLICA DA API (Utiliza esta no teu Front-end):
NgrokTunnel: "https://giggling-dormitory-tusk.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [09/Jul/2026 11:19:59] "GET / HTTP/1.1" 200 -
